# 12 Peak Picked Images

**Purpose.** Peak-picked image generation (threshold + rolling-window local maxima) before binning.

**Thesis reference.** §5 Representation, Not Model

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

DATA_ROOT = "../ahen_data/filtered_tsv_2"
LABELS    = "labels/labels_severity_all.json"
IMG       = 224
MZ_RANGE, RT_RANGE = (50.0, 1650.0), (0.0, 37.0)
THRESH, WINDOW = 5, 30          # df_find_peaks defaults
MZ_FLOOR  = 180                 # helper drops below this; set None to keep all
OCC_MASK  = True                # compact occupied m/z into the pixel columns
TIC_NORM  = True
MZ_BANDS  = 3
OUT = Path("images_picked")
for sub in ("grayscale_heatmap","rgb_mz_split"): (OUT/sub).mkdir(parents=True,exist_ok=True)

lab = json.load(open(LABELS))["labels"]
def npid(raw):
    d="".join(c for c in raw if c.isdigit()); return f"P{int(d):04d}" if d else None
files=[(f,npid(f.stem.replace("WCmsdata_s","")))
       for f in sorted(Path(DATA_ROOT).glob("*.tsv"))]
files=[(f,p) for f,p in files if p in lab]
print(f"{len(files)} files")

def read(path):
    parts=[]
    for ch in pd.read_csv(path,sep="\t",names=["time","mz","intensity"],
                          dtype=str,low_memory=False,chunksize=2_000_000):
        for c in ("time","mz","intensity"): ch[c]=pd.to_numeric(ch[c],errors="coerce")
        parts.append(ch.dropna())
    return pd.concat(parts,ignore_index=True)

def pick(df):
    df=df[df["intensity"]>THRESH].sort_values(["time","mz"])
    df=df.groupby("time",group_keys=False).apply(
        lambda x: x.loc[x["intensity"]==x["intensity"].rolling(
            WINDOW,center=True,min_periods=1).max()])
    if MZ_FLOOR: df=df[df["mz"]>=MZ_FLOOR]
    return df.reset_index(drop=True)

# pass 1: occupancy on PICKED peaks
MZ_FINE=8000
if OCC_MASK:
    print("pass 1: occupancy on picked peaks")
    occ=np.zeros(MZ_FINE,dtype=np.int32); t0=time.time()
    for k,(f,p) in enumerate(files,1):
        d=pick(read(f))
        m=(d.mz>=MZ_RANGE[0])&(d.mz<MZ_RANGE[1])
        idx=((d.mz[m]-MZ_RANGE[0])/(MZ_RANGE[1]-MZ_RANGE[0])*MZ_FINE).astype(int)
        occ[np.clip(idx,0,MZ_FINE-1)]+=1
        if k%50==0: print(f"  {k}/{len(files)}  {(time.time()-t0)/60:.0f} min")
    keep=occ>=0.05*len(files)
    edges=np.linspace(*MZ_RANGE,MZ_FINE+1); kept=edges[:-1][keep]
    print(f"  occupied fraction: {keep.mean():.1%}, {len(kept)} fine bins kept")
    np.save(OUT/"kept_mz.npy",kept)
    def col(v):
        return np.clip((np.searchsorted(kept,v)/max(len(kept),1)*IMG).astype(int),0,IMG-1)
    band_cuts=[(b[0],b[-1]) for b in np.array_split(np.sort(kept),MZ_BANDS)]
else:
    def col(v): return np.clip(((v-MZ_RANGE[0])/(MZ_RANGE[1]-MZ_RANGE[0])*IMG).astype(int),0,IMG-1)
    e=np.linspace(*MZ_RANGE,MZ_BANDS+1); band_cuts=list(zip(e[:-1],e[1:]))
def row(v): return np.clip(((v-RT_RANGE[0])/(RT_RANGE[1]-RT_RANGE[0])*IMG).astype(int),0,IMG-1)

# pass 2: write
print("pass 2: writing picked images")
t0=time.time(); occs=[]
for k,(f,p) in enumerate(files,1):
    d=pick(read(f))
    m=(d.mz>=MZ_RANGE[0])&(d.mz<MZ_RANGE[1])&(d.time>=RT_RANGE[0])&(d.time<RT_RANGE[1])
    d=d[m]
    mz,tm,it=d.mz.values,d.time.values,d.intensity.values
    if TIC_NORM and it.sum()>0: it=it/it.sum()*1e6
    c=col(mz); r=row(tm)
    H=np.zeros((IMG,IMG)); np.add.at(H,(r,c),it)
    occs.append((H>0).mean())
    H=np.log1p(H); H=H/(H.max() or 1)
    Image.fromarray((H*255).astype(np.uint8),"L").save(OUT/"grayscale_heatmap"/f"{p}.png")
    chans=[]
    for lo,hi in band_cuts:
        s=(mz>=lo)&(mz<=hi); Hc=np.zeros((IMG,IMG)); np.add.at(Hc,(r[s],c[s]),it[s])
        Hc=np.log1p(Hc); Hc=Hc/(Hc.max() or 1); chans.append((Hc*255).astype(np.uint8))
    Image.fromarray(np.stack(chans,-1),"RGB").save(OUT/"rgb_mz_split"/f"{p}.png")
    if k%50==0: print(f"  {k}/{len(files)}  {(time.time()-t0)/60:.0f} min")

print(f"\ndone in {(time.time()-t0)/60:.0f} min")
print(f"mean occupancy of picked images: {np.mean(occs):.1%}")
print(f"images in {OUT}/")
print(f"\nnext: Y_eval_both.py with IMAGE_ROOT = 'images_picked', TAG = 'picked'")
print("compare against 0.655 (images224) and 0.741 (map).")

510 files
pass 1: occupancy on picked peaks


/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecate

  50/510  29 min


/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecate

  100/510  62 min


/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecate

... (34 outputs left out)

  500/510  344 min


/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecate


done in 351 min
mean occupancy of picked images: 22.6%
images in images_picked/

next: Y_eval_both.py with IMAGE_ROOT = 'images_picked', TAG = 'picked'
compare against 0.655 (images224) and 0.741 (map).


/tmp/ipykernel_3150363/2311391787.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby("time",group_keys=False).apply(
